# Processamento de Língua Natural - EP

## SetFit com refinamento por hard negatives

O modelo classifica as respostas nas classes `c1`, `c234` e `c5`.

O treinamento possui duas etapas:

1. ajuste contrastivo inicial com `CosineSimilarityLoss`;
2. refinamento com exemplos difíceis identificados pelo próprio modelo.

Após o ajuste dos embeddings, uma regressão logística é treinada sobre as representações geradas pelo Sentence Transformer.

## Bibliotecas

No Google Colab, as dependências podem ser instaladas com:

```python
%pip install -q -U setfit==1.2.0 datasets sentence-transformers openpyxl scikit-learn "pandas==2.2.3"
```

Após a instalação, reinicie a sessão do Python antes de executar os imports.

In [ ]:
import gc
import random
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import (
    StratifiedKFold,
    train_test_split,
)

from sentence_transformers import (
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import (
    CachedMultipleNegativesRankingLoss,
    CosineSimilarityLoss,
)

from setfit import (
    SetFitModel,
    Trainer,
    TrainingArguments,
)

## Configuração

O backbone utilizado é o `paraphrase-multilingual-MiniLM-L12-v2`.

O ajuste inicial usa uma amostra estratificada de até 6.000 exemplos por fold. Em seguida, são selecionados exemplos que o modelo classificou incorretamente e são construídas triplas formadas por âncora, positivo e negativo difícil.

O refinamento é limitado a um conjunto pequeno de triplas para controlar o consumo de memória.

In [ ]:
SEED = 42

MODELO_BASE = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

N_SPLITS = 3
MAX_LENGTH = 64

# Ajuste contrastivo inicial.
EMBEDDING_EPOCHS = 1
BATCH_SIZE = 4
BODY_LEARNING_RATE = 2e-5
NUM_ITERATIONS = 1
CONTRASTIVE_SAMPLE_SIZE = 6000

# Refinamento com hard negatives.
HARD_TRIPLET_SIZE = 1500
HARD_EPOCHS = 1
HARD_BATCH_SIZE = 16
HARD_MINI_BATCH_SIZE = 4
HARD_LEARNING_RATE = 1e-5

# Classificador treinado sobre os embeddings.
HEAD_MAX_ITER = 1000
HEAD_CLASS_WEIGHT = "balanced"
HEAD_ENCODE_BATCH_SIZE = 32

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()

LOG_PATH = "resultados_setfit_hard_negatives.txt"


def fixar_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def liberar_memoria():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def log_resultado(mensagem=""):
    mensagem = str(mensagem)
    print(mensagem)

    with open(LOG_PATH, "a", encoding="utf-8") as arquivo:
        arquivo.write(mensagem + "\n")


fixar_seed(SEED)
liberar_memoria()

with open(LOG_PATH, "w", encoding="utf-8") as arquivo:
    arquivo.write("EP1 - SetFit com hard negatives\n")
    arquivo.write("=" * 64 + "\n")
    arquivo.write(
        f"Início: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    )
    arquivo.write(f"Modelo base: {MODELO_BASE}\n")
    arquivo.write(f"Dispositivo: {DEVICE}\n")
    arquivo.write(
        f"Ajuste inicial: {N_SPLITS} folds, "
        f"{EMBEDDING_EPOCHS} época(s), "
        f"MAX_LENGTH={MAX_LENGTH}, "
        f"BATCH_SIZE={BATCH_SIZE}, "
        f"CONTRASTIVE_SAMPLE_SIZE={CONTRASTIVE_SAMPLE_SIZE}\n"
    )
    arquivo.write(
        f"Refinamento: HARD_TRIPLET_SIZE={HARD_TRIPLET_SIZE}, "
        f"HARD_EPOCHS={HARD_EPOCHS}, "
        f"HARD_BATCH_SIZE={HARD_BATCH_SIZE}, "
        f"HARD_MINI_BATCH_SIZE={HARD_MINI_BATCH_SIZE}\n"
    )
    arquivo.write(
        f"Head: LogisticRegression, "
        f"class_weight={HEAD_CLASS_WEIGHT}\n"
    )
    arquivo.write("=" * 64 + "\n\n")

log_resultado(f"Dispositivo: {DEVICE}")
log_resultado(f"Modelo: {MODELO_BASE}")
log_resultado(f"Log: {LOG_PATH}")

## Importação dos dados

Os conjuntos de treino e teste são carregados a partir das planilhas fornecidas.

O texto é mantido próximo da forma original, sem remoção de stopwords, stemming ou lematização.

In [ ]:
dados_treino = pd.read_excel(
    "dados/train.xlsx",
    converters={"resp_text": str, "clarity": str},
)

dados_teste = pd.read_excel(
    "dados/test1.xlsx",
    converters={"resp_text": str, "clarity": str},
)

X_treino = (
    dados_treino["resp_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

Y_treino = (
    dados_treino["clarity"]
    .fillna("")
    .astype(str)
    .tolist()
)

X_teste = (
    dados_teste["resp_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

log_resultado()
log_resultado(f"Instâncias de treino: {len(X_treino)}")
log_resultado(f"Instâncias de teste: {len(X_teste)}")
log_resultado("Distribuição das classes no treino:")
log_resultado(
    dados_treino["clarity"]
    .value_counts()
    .to_string()
)

## Pré-processamento e rótulos

O pré-processamento normaliza apenas os espaços em branco.

Os rótulos `c1`, `c234` e `c5` são convertidos para valores inteiros durante o treinamento.

In [ ]:
def preprocess_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


LABEL_TO_INDEX = {
    "c1": 0,
    "c234": 1,
    "c5": 2,
}

INDEX_TO_LABEL = {
    0: "c1",
    1: "c234",
    2: "c5",
}

labels_invalidos = sorted(
    set(Y_treino) - set(LABEL_TO_INDEX)
)

if labels_invalidos:
    raise ValueError(
        f"Rótulos inesperados no treino: {labels_invalidos}"
    )

X_treino_processado = [
    preprocess_text(texto)
    for texto in X_treino
]

X_teste_processado = [
    preprocess_text(texto)
    for texto in X_teste
]

y_indices = np.array(
    [LABEL_TO_INDEX[label] for label in Y_treino],
    dtype=np.int64,
)

## Refinamento contrastivo

O primeiro ajuste usa pares gerados pelo SetFit com `CosineSimilarityLoss`.

Depois desse ajuste, o modelo é aplicado ao conjunto de treino do fold. Os exemplos classificados incorretamente são usados como âncoras. Para cada âncora:

- o positivo é o exemplo mais próximo da mesma classe;
- o negativo é o exemplo mais próximo da classe que o modelo confundiu com a classe correta.

As triplas resultantes são usadas em uma etapa curta de refinamento.

## Funções auxiliares

As funções abaixo criam os datasets, controlam a amostragem, ajustam a cabeça de classificação, constroem as triplas difíceis e executam as predições.

In [ ]:
def criar_dataset(textos, labels):
    return Dataset.from_dict(
        {
            "text": list(textos),
            "label": [
                int(label)
                for label in labels
            ],
        }
    )


def amostrar_treino(textos, labels, tamanho_maximo, seed):
    if len(textos) <= tamanho_maximo:
        return list(textos), np.asarray(labels, dtype=np.int64)

    indices_amostra, _ = train_test_split(
        np.arange(len(textos)),
        train_size=tamanho_maximo,
        stratify=labels,
        random_state=seed,
    )

    textos_amostra = [
        textos[i]
        for i in indices_amostra
    ]

    labels_amostra = np.asarray(labels)[indices_amostra]

    return textos_amostra, labels_amostra


def criar_modelo():
    model = SetFitModel.from_pretrained(
        MODELO_BASE,
        head_params={
            "max_iter": HEAD_MAX_ITER,
            "class_weight": HEAD_CLASS_WEIGHT,
            "random_state": SEED,
        },
    )

    model.model_body.max_seq_length = MAX_LENGTH
    model.to(DEVICE)

    return model


def criar_argumentos(seed):
    return TrainingArguments(
        batch_size=BATCH_SIZE,
        num_epochs=EMBEDDING_EPOCHS,
        num_iterations=NUM_ITERATIONS,
        body_learning_rate=BODY_LEARNING_RATE,
        loss=CosineSimilarityLoss,
        max_length=MAX_LENGTH,
        use_amp=USE_AMP,
        show_progress_bar=True,
        seed=seed,
        report_to="none",
    )


def prever_indices(model, textos):
    predicoes = model.predict(
        list(textos)
    )

    if torch.is_tensor(predicoes):
        predicoes = (
            predicoes
            .detach()
            .cpu()
            .numpy()
        )

    return np.asarray(
        predicoes,
        dtype=np.int64,
    )


def ajustar_head_completo(model, textos, labels):
    embeddings = model.model_body.encode(
        list(textos),
        batch_size=HEAD_ENCODE_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=model.normalize_embeddings,
    )

    model.model_head.fit(
        embeddings,
        np.asarray(labels, dtype=np.int64),
    )

    del embeddings
    liberar_memoria()


def selecionar_anchors_dificeis(
    labels_reais,
    labels_previstos,
    tamanho_maximo,
    seed,
):
    labels_reais = np.asarray(labels_reais)
    labels_previstos = np.asarray(labels_previstos)

    erros = np.flatnonzero(
        labels_reais != labels_previstos
    )

    if len(erros) <= tamanho_maximo:
        return erros

    rng = np.random.default_rng(seed)
    rng.shuffle(erros)

    return erros[:tamanho_maximo]


def minerar_triplets(
    model,
    textos,
    labels_reais,
    labels_previstos,
    tamanho_maximo,
    seed,
    chunk_size=64,
):
    labels_reais = np.asarray(
        labels_reais,
        dtype=np.int64,
    )

    labels_previstos = np.asarray(
        labels_previstos,
        dtype=np.int64,
    )

    anchors_idx = selecionar_anchors_dificeis(
        labels_reais,
        labels_previstos,
        tamanho_maximo,
        seed,
    )

    if len(anchors_idx) == 0:
        return Dataset.from_dict(
            {
                "anchor": [],
                "positive": [],
                "negative": [],
            }
        )

    embeddings = model.model_body.encode(
        list(textos),
        batch_size=HEAD_ENCODE_BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    anchors = []
    positives = []
    negatives = []

    for inicio in range(
        0,
        len(anchors_idx),
        chunk_size,
    ):
        lote_idx = anchors_idx[
            inicio:inicio + chunk_size
        ]

        similaridades = (
            embeddings[lote_idx]
            @ embeddings.T
        )

        for linha, anchor_idx in enumerate(lote_idx):
            classe_real = labels_reais[anchor_idx]
            classe_prevista = labels_previstos[anchor_idx]

            sims = similaridades[linha].copy()
            sims[anchor_idx] = -np.inf

            mascara_positiva = (
                labels_reais == classe_real
            )

            scores_positivos = np.where(
                mascara_positiva,
                sims,
                -np.inf,
            )

            positive_idx = int(
                np.argmax(scores_positivos)
            )

            mascara_negativa = (
                labels_reais == classe_prevista
            )

            if not np.any(mascara_negativa):
                mascara_negativa = (
                    labels_reais != classe_real
                )

            scores_negativos = np.where(
                mascara_negativa,
                sims,
                -np.inf,
            )

            negative_idx = int(
                np.argmax(scores_negativos)
            )

            if (
                not np.isfinite(
                    scores_positivos[positive_idx]
                )
                or not np.isfinite(
                    scores_negativos[negative_idx]
                )
            ):
                continue

            anchors.append(
                textos[anchor_idx]
            )

            positives.append(
                textos[positive_idx]
            )

            negatives.append(
                textos[negative_idx]
            )

        del similaridades

    del embeddings
    liberar_memoria()

    return Dataset.from_dict(
        {
            "anchor": anchors,
            "positive": positives,
            "negative": negatives,
        }
    )


def refinar_com_hard_negatives(
    model,
    dataset_triplets,
    seed,
    sufixo,
):
    if len(dataset_triplets) == 0:
        return

    loss_refino = (
        CachedMultipleNegativesRankingLoss(
            model.model_body,
            mini_batch_size=HARD_MINI_BATCH_SIZE,
        )
    )

    args_refino = (
        SentenceTransformerTrainingArguments(
            output_dir=f"tmp_setfit_{sufixo}",
            num_train_epochs=HARD_EPOCHS,
            per_device_train_batch_size=HARD_BATCH_SIZE,
            learning_rate=HARD_LEARNING_RATE,
            warmup_ratio=0.1,
            fp16=USE_AMP,
            bf16=False,
            save_strategy="no",
            eval_strategy="no",
            logging_steps=20,
            report_to="none",
            seed=seed,
        )
    )

    trainer_refino = SentenceTransformerTrainer(
        model=model.model_body,
        args=args_refino,
        train_dataset=dataset_triplets,
        loss=loss_refino,
    )

    trainer_refino.train()

    del trainer_refino
    del loss_refino
    liberar_memoria()

## Validação cruzada

A avaliação usa `StratifiedKFold` com 3 folds.

A mineração dos hard negatives usa somente dados pertencentes ao conjunto de treino de cada fold. A validação permanece separada durante todo o treinamento.

In [ ]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

resultados = []

predicoes_oof = np.full(
    len(X_treino_processado),
    -1,
    dtype=np.int64,
)

inicio_cv = time.time()

for fold, (train_idx, val_idx) in enumerate(
    cv.split(
        X_treino_processado,
        y_indices,
    ),
    start=1,
):
    liberar_memoria()
    fixar_seed(SEED + fold)

    inicio_fold = time.time()

    log_resultado()
    log_resultado("=" * 64)
    log_resultado(f"FOLD {fold}/{N_SPLITS}")
    log_resultado("=" * 64)

    textos_fold_treino = [
        X_treino_processado[i]
        for i in train_idx
    ]

    textos_fold_validacao = [
        X_treino_processado[i]
        for i in val_idx
    ]

    y_fold_treino = y_indices[train_idx]
    y_fold_validacao = y_indices[val_idx]

    textos_contrastivos, y_contrastivos = amostrar_treino(
        textos_fold_treino,
        y_fold_treino,
        tamanho_maximo=CONTRASTIVE_SAMPLE_SIZE,
        seed=SEED + fold,
    )

    log_resultado(
        f"Exemplos no ajuste inicial: "
        f"{len(textos_contrastivos)}"
    )

    dataset_fold_treino = criar_dataset(
        textos_contrastivos,
        y_contrastivos,
    )

    model = criar_modelo()

    args = criar_argumentos(
        seed=SEED + fold
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset_fold_treino,
        metric="accuracy",
    )

    trainer.train()

    # A cabeça usa todas as instâncias do fold de treino.
    ajustar_head_completo(
        model,
        textos_fold_treino,
        y_fold_treino,
    )

    pred_treino_inicial = prever_indices(
        model,
        textos_fold_treino,
    )

    pred_validacao_inicial = prever_indices(
        model,
        textos_fold_validacao,
    )

    acc_validacao_inicial = accuracy_score(
        y_fold_validacao,
        pred_validacao_inicial,
    )

    log_resultado(
        f"Validação antes do refino: "
        f"{acc_validacao_inicial:.2%}"
    )

    dataset_triplets = minerar_triplets(
        model=model,
        textos=textos_fold_treino,
        labels_reais=y_fold_treino,
        labels_previstos=pred_treino_inicial,
        tamanho_maximo=HARD_TRIPLET_SIZE,
        seed=SEED + fold,
    )

    log_resultado(
        f"Triplets difíceis: "
        f"{len(dataset_triplets)}"
    )

    refinar_com_hard_negatives(
        model=model,
        dataset_triplets=dataset_triplets,
        seed=SEED + fold,
        sufixo=f"fold_{fold}",
    )

    # O refinamento altera os embeddings.
    # A cabeça é ajustada novamente sobre o fold inteiro.
    ajustar_head_completo(
        model,
        textos_fold_treino,
        y_fold_treino,
    )

    pred_treino = prever_indices(
        model,
        textos_fold_treino,
    )

    pred_validacao = prever_indices(
        model,
        textos_fold_validacao,
    )

    predicoes_oof[val_idx] = pred_validacao

    acc_treino = accuracy_score(
        y_fold_treino,
        pred_treino,
    )

    acc_validacao = accuracy_score(
        y_fold_validacao,
        pred_validacao,
    )

    tempo_fold = (
        time.time() - inicio_fold
    )

    resultados.append(
        {
            "fold": fold,
            "validation_before_refinement": acc_validacao_inicial,
            "train_accuracy": acc_treino,
            "validation_accuracy": acc_validacao,
            "tempo_segundos": tempo_fold,
        }
    )

    log_resultado(
        f"Acurácia de treino: "
        f"{acc_treino:.2%}"
    )

    log_resultado(
        f"Acurácia de validação: "
        f"{acc_validacao:.2%}"
    )

    log_resultado(
        f"Ganho do refino: "
        f"{acc_validacao - acc_validacao_inicial:+.2%}"
    )

    log_resultado(
        f"Gap do fold: "
        f"{acc_treino - acc_validacao:.2%}"
    )

    log_resultado(
        f"Tempo do fold: "
        f"{tempo_fold / 60:.2f} min"
    )

    del trainer
    del model
    del dataset_fold_treino
    del dataset_triplets
    del textos_fold_treino
    del textos_fold_validacao
    del textos_contrastivos
    del y_fold_treino
    del y_fold_validacao
    del y_contrastivos
    del pred_treino_inicial
    del pred_validacao_inicial

    liberar_memoria()

resultados_df = pd.DataFrame(
    resultados
)

acuracia_validacao_inicial = (
    resultados_df[
        "validation_before_refinement"
    ].mean()
)

acuracia_validacao = (
    resultados_df[
        "validation_accuracy"
    ].mean()
)

desvio_validacao = (
    resultados_df[
        "validation_accuracy"
    ].std()
)

acuracia_treino_cv = (
    resultados_df[
        "train_accuracy"
    ].mean()
)

gap = (
    acuracia_treino_cv
    - acuracia_validacao
)

tempo_cv = (
    time.time() - inicio_cv
)

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "RESUMO DA VALIDAÇÃO CRUZADA"
)
log_resultado("=" * 64)

log_resultado(
    resultados_df.to_string(
        index=False
    )
)

log_resultado()

log_resultado(
    f"Validação média antes do refino: "
    f"{acuracia_validacao_inicial:.2%}"
)

log_resultado(
    f"Validação média depois do refino: "
    f"{acuracia_validacao:.2%}"
)

log_resultado(
    f"Ganho médio do refino: "
    f"{acuracia_validacao - acuracia_validacao_inicial:+.2%}"
)

log_resultado(
    f"Desvio-padrão da validação: "
    f"{desvio_validacao:.2%}"
)

log_resultado(
    f"Acurácia média de treino: "
    f"{acuracia_treino_cv:.2%}"
)

log_resultado(
    f"Gap treino-validação: "
    f"{gap:.2%}"
)

log_resultado(
    f"Tempo total da validação cruzada: "
    f"{tempo_cv / 60:.2f} min"
)

## Matriz de confusão e relatório OOF

A matriz de confusão e o `classification_report` são calculados a partir das previsões OOF produzidas depois do refinamento.

In [ ]:
ordem_classes = [0, 1, 2]
nomes_classes = [
    "c1",
    "c234",
    "c5",
]

matriz = confusion_matrix(
    y_indices,
    predicoes_oof,
    labels=ordem_classes,
)

relatorio = classification_report(
    y_indices,
    predicoes_oof,
    labels=ordem_classes,
    target_names=nomes_classes,
    digits=4,
    zero_division=0,
)

matriz_df = pd.DataFrame(
    matriz,
    index=[
        f"real_{classe}"
        for classe in nomes_classes
    ],
    columns=[
        f"pred_{classe}"
        for classe in nomes_classes
    ],
)

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "MATRIZ DE CONFUSÃO OOF"
)
log_resultado("=" * 64)
log_resultado(
    matriz_df.to_string()
)

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "CLASSIFICATION REPORT OOF"
)
log_resultado("=" * 64)
log_resultado(relatorio)

matriz_df

## Confusões entre classes

Os erros são agrupados pelos pares:

- `c1` e `c234`;
- `c234` e `c5`;
- `c1` e `c5`.

In [ ]:
def contar_confusao(real, previsto, classe_a, classe_b):
    a = LABEL_TO_INDEX[classe_a]
    b = LABEL_TO_INDEX[classe_b]

    return int(
        (
            ((real == a) & (previsto == b))
            |
            ((real == b) & (previsto == a))
        ).sum()
    )


confusoes_fronteira = {
    "c1 <-> c234": contar_confusao(
        y_indices,
        predicoes_oof,
        "c1",
        "c234",
    ),
    "c234 <-> c5": contar_confusao(
        y_indices,
        predicoes_oof,
        "c234",
        "c5",
    ),
    "c1 <-> c5": contar_confusao(
        y_indices,
        predicoes_oof,
        "c1",
        "c5",
    ),
}

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "CONFUSÕES ENTRE PARES DE CLASSES"
)
log_resultado("=" * 64)

for par, quantidade in confusoes_fronteira.items():
    log_resultado(
        f"{par}: {quantidade}"
    )

confusoes_fronteira

## Treinamento final

O ajuste contrastivo inicial utiliza uma amostra estratificada de até 6.000 exemplos.

Os hard negatives são minerados somente no conjunto de treino rotulado. Depois do refinamento, a regressão logística é ajustada utilizando todo o conjunto de treino.

In [ ]:
liberar_memoria()
fixar_seed(SEED)

inicio_treino_final = time.time()

textos_treino_inicial, y_treino_inicial = amostrar_treino(
    X_treino_processado,
    y_indices,
    tamanho_maximo=CONTRASTIVE_SAMPLE_SIZE,
    seed=SEED,
)

log_resultado()
log_resultado("=" * 64)
log_resultado("TREINO FINAL")
log_resultado("=" * 64)

log_resultado(
    f"Exemplos no ajuste inicial final: "
    f"{len(textos_treino_inicial)}"
)

dataset_treino_final = criar_dataset(
    textos_treino_inicial,
    y_treino_inicial,
)

model_final = criar_modelo()

args_final = criar_argumentos(
    seed=SEED
)

trainer_final = Trainer(
    model=model_final,
    args=args_final,
    train_dataset=dataset_treino_final,
    metric="accuracy",
)

trainer_final.train()

ajustar_head_completo(
    model_final,
    X_treino_processado,
    y_indices,
)

pred_treino_inicial_final = prever_indices(
    model_final,
    X_treino_processado,
)

dataset_triplets_final = minerar_triplets(
    model=model_final,
    textos=X_treino_processado,
    labels_reais=y_indices,
    labels_previstos=pred_treino_inicial_final,
    tamanho_maximo=HARD_TRIPLET_SIZE,
    seed=SEED,
)

log_resultado(
    f"Triplets difíceis no treino final: "
    f"{len(dataset_triplets_final)}"
)

refinar_com_hard_negatives(
    model=model_final,
    dataset_triplets=dataset_triplets_final,
    seed=SEED,
    sufixo="final",
)

ajustar_head_completo(
    model_final,
    X_treino_processado,
    y_indices,
)

pred_treino_final = prever_indices(
    model_final,
    X_treino_processado,
)

acc_treino_final = accuracy_score(
    y_indices,
    pred_treino_final,
)

tempo_treino_final = (
    time.time() - inicio_treino_final
)

log_resultado(
    f"Acurácia do treino (modelo final): "
    f"{acc_treino_final:.2%}"
)

log_resultado(
    f"Tempo do treino final: "
    f"{tempo_treino_final / 60:.2f} min"
)

del dataset_treino_final
del dataset_triplets_final
del textos_treino_inicial
del y_treino_inicial
del pred_treino_inicial_final

liberar_memoria()

## Predição do conjunto de teste

As classes previstas são convertidas novamente para `c1`, `c234` e `c5`, preservando a ordem das linhas de `test1.xlsx`.

In [ ]:
pred_teste_indices = prever_indices(
    model_final,
    X_teste_processado,
)

Y_previsto = [
    INDEX_TO_LABEL[int(indice)]
    for indice in pred_teste_indices
]

assert len(Y_previsto) == len(X_teste)

assert set(Y_previsto).issubset(
    {"c1", "c234", "c5"}
)

distribuicao_teste = (
    pd.Series(Y_previsto)
    .value_counts()
)

log_resultado()
log_resultado("=" * 64)
log_resultado(
    "DISTRIBUIÇÃO DAS PREVISÕES DO TESTE"
)
log_resultado("=" * 64)
log_resultado(
    distribuicao_teste.to_string()
)

distribuicao_teste

In [ ]:
teste_rotulado = pd.DataFrame(
    {
        "resp_text": X_teste,
        "clarity": Y_previsto,
    }
)

teste_rotulado.to_excel(
    "teste_rotulado_setfit_hard_negatives.xlsx",
    index=False,
)

oof_df = pd.DataFrame(
    {
        "resp_text": X_treino,
        "clarity_real": Y_treino,
        "clarity_pred_oof": [
            INDEX_TO_LABEL[int(indice)]
            for indice in predicoes_oof
        ],
    }
)

oof_df["acertou"] = (
    oof_df["clarity_real"]
    == oof_df["clarity_pred_oof"]
)

oof_df.to_csv(
    "predicoes_oof_setfit_hard_negatives.csv",
    index=False,
    encoding="utf-8",
)

log_resultado()
log_resultado(
    "Arquivo de teste salvo: "
    "teste_rotulado_setfit_hard_negatives.xlsx"
)

log_resultado(
    "Predições OOF salvas: "
    "predicoes_oof_setfit_hard_negatives.csv"
)

log_resultado(
    f"Arquivo de log salvo: {LOG_PATH}"
)

log_resultado(
    f"Fim: "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

teste_rotulado.head()

## Registro dos resultados

A célula abaixo exibe o conteúdo salvo em `resultados_setfit_hard_negatives.txt`.

In [ ]:
with open(
    LOG_PATH,
    "r",
    encoding="utf-8",
) as arquivo:
    print(arquivo.read())